In [4]:
import torch
from torch.utils.data import Dataset

class ABSADataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len=128):
        self.df = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.loc[idx]

        sentence = row["Sentence"]
        aspect = row["Aspect Term"]
        label = LABEL_MAP[row["polarity"]]

        encoding = self.tokenizer(
            sentence,
            aspect,
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt"
        )

        item = {k: v.squeeze(0) for k, v in encoding.items()}
        item["labels"] = torch.tensor(label, dtype=torch.long)

        return item


In [14]:
import torch
import pandas as pd
from torch.utils.data import DataLoader
from torch.optim import AdamW

from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    get_linear_schedule_with_warmup
)

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

# --------------------
# Config
# --------------------
MODEL_NAME = "bert-base-uncased"
MAX_LEN = 128
BATCH_SIZE = 32
EPOCHS = 10
LR = 2e-5
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

LABEL_MAP = {
    "negative": 0,
    "neutral": 1,
    "positive": 2,
    "conflict": 3
}


# --------------------
# Load Data
# --------------------
df = pd.read_csv(
    r"D:\Sem6\NLP\absa-nlp\data\raw\Laptop_Train_v2.csv"
)

train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["polarity"]
)

tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)

train_dataset = ABSADataset(train_df, tokenizer)
val_dataset = ABSADataset(val_df, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

# --------------------
# Model
# --------------------
model = BertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=4
)
model.to(DEVICE)

# --------------------
# Optimizer & Scheduler
# --------------------
optimizer = AdamW(model.parameters(), lr=LR)

total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,
    num_training_steps=total_steps
)

# --------------------
# Training Loop
# --------------------
def train_epoch(model, loader):
    model.train()
    total_loss = 0

    for batch in loader:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}

        optimizer.zero_grad()
        outputs = model(**batch)
        loss = outputs.loss

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()

    return total_loss / len(loader)

# --------------------
# Evaluation
# --------------------
def evaluate(model, loader):
    model.eval()
    preds, labels = [], []

    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            outputs = model(**batch)

            logits = outputs.logits
            preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
            labels.extend(batch["labels"].cpu().numpy())

    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average="macro")
    return acc, f1

# --------------------
# Run Training
# --------------------
for epoch in range(EPOCHS):
    loss = train_epoch(model, train_loader)
    acc, f1 = evaluate(model, val_loader)

    print(f"Epoch {epoch+1}/{EPOCHS}")
    print(f"Loss: {loss:.4f} | Val Acc: {acc:.4f} | Val F1: {f1:.4f}")

# --------------------
# Save
# --------------------
model.save_pretrained("bert_absa_model")
tokenizer.save_pretrained("bert_absa_model")


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch 1/10
Loss: 1.1346 | Val Acc: 0.6928 | Val F1: 0.4056
Epoch 2/10
Loss: 0.7498 | Val Acc: 0.7691 | Val F1: 0.5485
Epoch 3/10
Loss: 0.5415 | Val Acc: 0.7775 | Val F1: 0.5645
Epoch 4/10
Loss: 0.4167 | Val Acc: 0.7924 | Val F1: 0.5840
Epoch 5/10
Loss: 0.3327 | Val Acc: 0.7797 | Val F1: 0.5721
Epoch 6/10
Loss: 0.2581 | Val Acc: 0.7669 | Val F1: 0.5643
Epoch 7/10
Loss: 0.2021 | Val Acc: 0.7733 | Val F1: 0.5698
Epoch 8/10
Loss: 0.1673 | Val Acc: 0.7818 | Val F1: 0.5768
Epoch 9/10
Loss: 0.1405 | Val Acc: 0.7669 | Val F1: 0.5666
Epoch 10/10
Loss: 0.1188 | Val Acc: 0.7712 | Val F1: 0.5685


('bert_absa_model\\tokenizer_config.json',
 'bert_absa_model\\special_tokens_map.json',
 'bert_absa_model\\vocab.txt',
 'bert_absa_model\\added_tokens.json')

In [19]:
import torch
from transformers import BertTokenizer, BertForSequenceClassification

# --------------------
# Config
# --------------------
MODEL_PATH = "bert_absa_model"   # change if different
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Label mapping (MUST match training)
ID2LABEL = {
    0: "negative",
    1: "neutral",
    2: "positive",
    3: "conflict"
}

# --------------------
# Load model & tokenizer
# --------------------
tokenizer = BertTokenizer.from_pretrained(MODEL_PATH)
model = BertForSequenceClassification.from_pretrained(MODEL_PATH)
model.to(DEVICE)
model.eval()

# --------------------
# Inference function
# --------------------
def predict_aspect_sentiment(sentence, aspect):
    inputs = tokenizer(
        sentence,
        aspect,
        return_tensors="pt",
        truncation=True,
        max_length=128
    )

    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        pred_id = torch.argmax(logits, dim=1).item()

    return ID2LABEL[pred_id]

# --------------------
# Example test
# --------------------
sentence = "I charge it at night and skip taking the cord with me because of the good battery life."
aspect = "battery life"

prediction = predict_aspect_sentiment(sentence, aspect)
print("Sentence:", sentence)
print("Aspect:", aspect)
print("Predicted Sentiment:", prediction)


Sentence: I charge it at night and skip taking the cord with me because of the good battery life.
Aspect: battery life
Predicted Sentiment: positive


In [18]:
import torch
from transformers import BertTokenizer, BertForSequenceClassification

# --------------------
# Config
# --------------------
MODEL_PATH = "bert_absa_model"   # change if different
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Label mapping (MUST match training)
ID2LABEL = {
    0: "negative",
    1: "neutral",
    2: "positive",
    3: "conflict"
}

# --------------------
# Load model & tokenizer
# --------------------
tokenizer = BertTokenizer.from_pretrained(MODEL_PATH)
model = BertForSequenceClassification.from_pretrained(MODEL_PATH)
model.to(DEVICE)
model.eval()

# --------------------
# Inference function
# --------------------
def predict_aspect_sentiment(sentence, aspect):
    inputs = tokenizer(
        sentence,
        aspect,
        return_tensors="pt",
        truncation=True,
        max_length=128
    )

    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        pred_id = torch.argmax(logits, dim=1).item()

    return ID2LABEL[pred_id]

# --------------------
# Example test
# --------------------
sentence = "My laptop display is awesome, but the battery life is terrible."
aspect = "display"

prediction = predict_aspect_sentiment(sentence, aspect)
print("Sentence:", sentence)
print("Aspect:", aspect)
print("Predicted Sentiment:", prediction)


Sentence: My laptop display is awesome, but the battery life is terrible.
Aspect: display
Predicted Sentiment: positive
